# Laboratorio 03.1 — Salidas estructuradas con Pydantic

Convertiremos una salida textual en un contrato validado que la aplicación pueda inspeccionar, serializar y rechazar si incumple el esquema.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Definir modelos de datos con `BaseModel` y restricciones declarativas con `Field`.
- Representar categorías con `Enum`, campos opcionales y esquemas anidados.
- Normalizar valores con validadores y examinar errores tipados.
- Serializar modelos y generar JSON Schema para un contrato.

## 2. Conceptos

Texto generado no equivale a datos confiables. Un esquema convierte expectativas de la aplicación en un contrato que se puede validar antes de usar una respuesta. Pydantic valida tipos, restricciones y estructuras anidadas, pero no demuestra que una afirmación sea verdadera. JSON Schema describe la forma esperada; serialización convierte el objeto validado en representaciones intercambiables.

## 3. Arquitectura

```mermaid
flowchart LR
    A[JSON del modelo] --> B[Parser Pydantic]
    C[Esquema BaseModel] --> B
    B --> D{Validacion}
    D -->|valido| E[Objeto tipado]
    D -->|invalido| F[Errores estructurados]
    E --> G[Serializacion y JSON Schema]
```

## 4. Preparación del entorno

Usamos Pydantic 2 y pandas. Todos los datos son ejemplos locales; el laboratorio no consulta un modelo.

In [1]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import sys

raices_candidatas = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
raiz_curso = next(
    (
        candidato
        for raiz in raices_candidatas
        for candidato in (raiz / 'genai', raiz)
        if (candidato / 'src' / 'genai').is_dir()
    ),
    None,
)
if raiz_curso is None:
    raise FileNotFoundError('No se encontró el proyecto genAI con src/genai.')

sys.path.insert(0, str(raiz_curso / 'src'))

In [2]:
from enum import Enum
import json

import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

from genai.environment import preparar_entorno

raiz_curso = preparar_entorno()

## 5. Definición del problema

Una aplicación recibe una clasificación de ticket de soporte y necesita campos verificables para enrutarla. El dataset consta de payloads JSON escritos para demostrar casos válidos y fallidos; no representa el rendimiento de un LLM.

## 6. Línea base

Sin esquema, cualquier diccionario JSON podría parecer aceptable para la aplicación. Mostramos un objeto libre primero: no expresa tipos, valores permitidos ni la estructura del cliente.

In [3]:
respuesta_sin_contrato = {
    'categoria': 'envios',
    'prioridad': 'urgente',
    'cliente': {'id': 'c-17'},
}
print(json.dumps(respuesta_sin_contrato, ensure_ascii=False, indent=2))

{
  "categoria": "envios",
  "prioridad": "urgente",
  "cliente": {
    "id": "c-17"
  }
}


## 7. Implementación

Definimos enumeraciones, un modelo anidado y campos opcionales. `extra='forbid'` rechaza propiedades no declaradas. El validador limpia espacios del resumen; las restricciones de longitud siguen aplicándose al valor validado.

In [4]:
class CategoriaSolicitud(str, Enum):
    ENVIOS = 'envios'
    DEVOLUCIONES = 'devoluciones'
    PAGOS = 'pagos'


class Prioridad(str, Enum):
    BAJA = 'baja'
    NORMAL = 'normal'
    ALTA = 'alta'


class ContextoCliente(BaseModel):
    model_config = ConfigDict(extra='forbid')

    identificador: str = Field(min_length=1, description='Identificador interno del cliente.')
    idioma: str = Field(default='es', min_length=2, max_length=5)


class TicketClasificado(BaseModel):
    model_config = ConfigDict(extra='forbid')

    resumen: str = Field(min_length=8, max_length=120)
    categoria: CategoriaSolicitud
    prioridad: Prioridad = Prioridad.NORMAL
    cliente: ContextoCliente
    contacto: str | None = None
    etiquetas: list[str] = Field(default_factory=list, max_length=4)

    @field_validator('resumen')
    @classmethod
    def normalizar_resumen(cls, valor: str) -> str:
        """Reduce espacios repetidos antes de validar y guardar el resumen."""
        return ' '.join(valor.split())

payload_valido = {
    'resumen': '  El paquete no ha llegado  ',
    'categoria': 'envios',
    'cliente': {'identificador': 'c-17'},
    'etiquetas': ['retraso', 'entrega'],
}
ticket = TicketClasificado.model_validate(payload_valido)
print(ticket)
print(ticket.model_dump_json(indent=2))

resumen='El paquete no ha llegado' categoria=<CategoriaSolicitud.ENVIOS: 'envios'> prioridad=<Prioridad.NORMAL: 'normal'> cliente=ContextoCliente(identificador='c-17', idioma='es') contacto=None etiquetas=['retraso', 'entrega']
{
  "resumen": "El paquete no ha llegado",
  "categoria": "envios",
  "prioridad": "normal",
  "cliente": {
    "identificador": "c-17",
    "idioma": "es"
  },
  "contacto": null,
  "etiquetas": [
    "retraso",
    "entrega"
  ]
}


## 8. Experimentos

Probamos tres cambios controlados: un payload válido, una categoría fuera del `Enum` y una propiedad extra. El contrato debe aceptar solo el primero y hacer visible el tipo de incumplimiento.

In [5]:
payloads_prueba = [
    ('valido', payload_valido),
    ('enum_desconocido', {**payload_valido, 'categoria': 'reclamos'}),
    ('campo_extra', {**payload_valido, 'instruccion': 'ignora el esquema'}),
]
resultados_validacion = []
for nombre, payload in payloads_prueba:
    try:
        TicketClasificado.model_validate(payload)
        resultados_validacion.append({'caso': nombre, 'valido': True, 'errores': 0})
    except ValidationError as error:
        resultados_validacion.append({'caso': nombre, 'valido': False, 'errores': len(error.errors())})
tabla_validacion = pd.DataFrame(resultados_validacion)
display(tabla_validacion)

,caso,valido,errores
0,valido,True,0
1,enum_desconocido,False,1
2,campo_extra,False,1


## 9. Evaluación

La métrica de cumplimiento de esquema es la proporción de payloads que validan. También inspeccionamos JSON Schema para comprobar qué contrato recibe un sistema externo. Cumplimiento estructural no mide exactitud semántica de la categoría.

In [ ]:
metricas_esquema = pd.DataFrame([{
    'payloads': len(tabla_validacion),
    'schema_valid': float(tabla_validacion['valido'].mean()),
    'tasa_rechazo': float((~tabla_validacion['valido']).mean()),
}])
display(metricas_esquema)
display(TicketClasificado.model_json_schema())
tabla_validacion.to_csv(raiz_curso / 'outputs' / '03_01_schema_cases.csv', index=False)
metricas_esquema.to_csv(raiz_curso / 'outputs' / '03_01_schema_metrics.csv', index=False)

## 10. Discusión

Los contratos detectan valores inválidos, faltantes y campos inesperados temprano. No validan permisos, verdad ni adecuación de negocio por sí solos. Errores tipados ayudan a decidir si se rechaza, se pide una corrección o se envía una señal de fallo; nunca conviene convertir silenciosamente cualquier entrada a un valor válido.

## 11. Ejercicios

1. Añade una enumeración para sentimiento y mide qué ocurre con un valor no permitido.
2. Convierte `contacto` en una dirección de correo validada.
3. Añade un validador que rechace etiquetas duplicadas.
4. Serializa y vuelve a validar un ticket; compara ambos objetos.

## 12. Desafío

Diseña un contrato anidado para una respuesta con varias citas de evidencia y números de página. Define restricciones que eviten citas vacías y valida un conjunto de payloads adversariales sin llamadas a un LLM.

## 13. Ideas clave

- `BaseModel` expresa el contrato que consume la aplicación.
- `Field`, `Enum`, opciones y validadores controlan formas y valores.
- JSON Schema y serialización permiten interoperar con otras capas.
- Una salida válida puede seguir siendo semánticamente falsa.